# Two-Model Reference

This notebook runs the canonical two-model reference configuration at `N=8` with the larger `10000/2000` train/validation split. The reusable implementation lives in `src/workflows/` and `src/models/two_models/`.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from config import Stage1ModelConfig, Stage2ModelConfig, StageTrainingConfig, TwoStageRunConfig, TwoStageStackConfig
from pipeline.run import run_two_stage_once
from models import set_torch_seed

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
np.random.seed(SEED)
set_torch_seed(SEED)

OUTPUT_ROOT = ROOT / 'outputs' / 'two_models'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

N_values = [8]
training_sizes = [10000]
validation_sizes = [2000]

stage1_config = Stage1ModelConfig(
    hidden_layer_sizes=(256, 512, 256),
    dropout_rates=(0.15, 0.15, 0.15),
    training=StageTrainingConfig(
        epochs=220,
        batch_size=64,
        learning_rate=0.0004,
        validation_frequency=10,
        verbose=False,
        early_stopping_patience=12,
        lr_drop_factor=0.5,
        lr_drop_period=60,
        weight_decay=0.00001,
        gradient_clip_norm=1.0,
    ),
)

stage2_config = Stage2ModelConfig(
    hidden_layer_sizes=(512, 1024),
    dropout_rates=(0.10, 0.10),
    model_type='coord_conv_decoder',
    latent_grid_size=16,
    latent_channels=160,
    decoder_channels=(160, 128, 96, 64, 32),
    use_rectangle_edge_weighting=True,
    use_foreground_pos_weight=False,
    rectangle_edge_weight=4.0,
    rectangle_edge_width=3,
    edge_weight_mode='rectangle',
    annulus_edge_weight=1.0,
    annulus_edge_width=3,
    training=StageTrainingConfig(
        epochs=170,
        batch_size=96,
        learning_rate=0.0005,
        validation_frequency=60,
        verbose=False,
        early_stopping_patience=24,
        min_epochs=50,
        min_improvement=0.002,
        lr_drop_factor=0.5,
        lr_drop_period=80,
        weight_decay=0.00025,
        gradient_clip_norm=0.8,
        loss_type='bce_dice',
        dice_loss_weight=1.0,
        dice_smooth=1.0,
    ),
)

print('Device:', DEVICE)
print('Output root:', OUTPUT_ROOT)
print('Sweep:', N_values, training_sizes, validation_sizes)
print('Stage 1:', stage1_config)
print('Stage 2:', stage2_config)

def plot_history(history: dict, title: str, output_path: Path):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(np.arange(1, len(history['train_loss']) + 1), history['train_loss'], marker='o')
    axes[0].set_title(f'{title} Train Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.3)
    axes[1].plot(history['validation_steps'], history['val_loss'], marker='o', color='tab:orange')
    axes[1].set_title(f'{title} Validation Loss')
    axes[1].set_xlabel('Validation Step')
    axes[1].set_ylabel('Loss')
    axes[1].grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

Device: cuda
Output root: c:\Users\prest\Documents\minimal-gravimetry-ml\outputs\two_models
Sweep: [8] [10000] [2000]
Stage 1: Stage1ModelConfig(hidden_layer_sizes=(256, 512, 256), dropout_rates=(0.15, 0.15, 0.15), training=StageTrainingConfig(epochs=220, batch_size=64, learning_rate=0.0004, validation_frequency=10, verbose=False, early_stopping_patience=12, min_epochs=None, min_improvement=None, lr_drop_factor=0.5, lr_drop_period=60, weight_decay=1e-05, gradient_clip_norm=1.0, loss_type='bce', dice_loss_weight=0.0, dice_smooth=1.0))
Stage 2: Stage2ModelConfig(hidden_layer_sizes=(512, 1024), dropout_rates=(0.1, 0.1), model_type='coord_conv_decoder', latent_grid_size=16, latent_channels=160, decoder_channels=(160, 128, 96, 64, 32), use_rectangle_edge_weighting=True, use_foreground_pos_weight=False, rectangle_edge_weight=4.0, rectangle_edge_width=3, edge_weight_mode='rectangle', annulus_edge_weight=1.0, annulus_edge_width=3, training=StageTrainingConfig(epochs=170, batch_size=96, learnin

In [ ]:
def make_run_config(index: int, n_value: int) -> TwoStageRunConfig:
    return TwoStageRunConfig(
        N=n_value,
        training_samples=training_sizes[index],
        validation_samples=validation_sizes[index],
        test_samples=500,
        rho=0.8,
        grid_size=32,
        threshold=0.5,
        use_validation_threshold_sweep=True,
        noise_sigma=0.01,
        seed=SEED + index,
        training_shape_weights=(
            ('two_circles', 0.40),
            ('annulus', 0.20),
            ('rectangle', 0.20),
            ('ellipse', 0.10),
            ('circle', 0.10),
        ),
        model=TwoStageStackConfig(stage1=stage1_config, stage2=stage2_config),
        output_dir=OUTPUT_ROOT,
    )


sweep_rows = []
for index, n_value in enumerate(N_values):
    run_config = make_run_config(index, n_value)
    print(f'Running N={n_value} -> {run_config.run_output_dir}')
    summary = run_two_stage_once(run_config, device=DEVICE)
    plot_history(summary['stage1_history'], 'Task 8 Stage 1', run_config.run_output_dir / 'stage1_losses.png')
    plot_history(summary['stage2_history'], 'Final Two-Stage Decoder Weight Adjusted', run_config.run_output_dir / 'general_losses.png')
    sweep_rows.append(summary)

with (OUTPUT_ROOT / 'sweep_summary.json').open('w', encoding='utf-8') as handle:
    json.dump({'runs': sweep_rows}, handle, indent=2)

test_ious = [row['metrics']['stage2_test']['mean_iou'] for row in sweep_rows]
fixed_ious = [row['metrics']['stage2_fixed']['mean_iou'] for row in sweep_rows]
condition_numbers = [row['condition_number'] for row in sweep_rows]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(N_values, test_ious, marker='o')
axes[0].set_title('Final Decoder Test Mean IoU')
axes[0].set_xlabel('N')
axes[0].set_ylabel('Mean IoU')
axes[0].grid(True, alpha=0.3)
axes[1].plot(N_values, fixed_ious, marker='o', color='tab:green')
axes[1].set_title('Final Decoder Fixed Mean IoU')
axes[1].set_xlabel('N')
axes[1].set_ylabel('Mean IoU')
axes[1].grid(True, alpha=0.3)
axes[2].plot(N_values, condition_numbers, marker='o', color='tab:red')
axes[2].set_title('Condition Number vs N')
axes[2].set_xlabel('N')
axes[2].set_ylabel('Cond(M)')
axes[2].grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / 'sweep_plots.png', dpi=150, bbox_inches='tight')
plt.show()

[{
    'N': row['config']['N'],
    'stage2_test_iou': row['metrics']['stage2_test']['mean_iou'],
    'stage2_fixed_iou': row['metrics']['stage2_fixed']['mean_iou'],
    'annulus_test_iou': row['metrics_by_shape']['stage2_test']['annulus']['mean_iou'],
    'rectangle_test_iou': row['metrics_by_shape']['stage2_test']['rectangle']['mean_iou'],
    'ellipse_test_iou': row['metrics_by_shape']['stage2_test']['ellipse']['mean_iou'],
    'two_circles_test_iou': row['metrics_by_shape']['stage2_test']['two_circles']['mean_iou'],
    'annulus_center_mean': row['diagnostics']['annulus_center']['predicted_coefficients']['center_mean_probability'],
} for row in sweep_rows]

Running N=8 -> c:\Users\prest\Documents\minimal-gravimetry-ml\outputs\two_models\N_8
